# Peruvian Amazon — Illegal Activity Detection: Results Analysis

Post-processing and visualisation of GEE pipeline outputs.  
Input: `alert_sites_madre_de_dios_2026_04_20.csv`  
Study area: Madre de Dios focus region, Peruvian Amazon  
200 RADD alert sites inside protected areas

In [3]:
!pip install pandas matplotlib numpy --break-system-packages

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import json

# Load CSV
df = pd.read_csv('alert_sites_madre_de_dios_2026_04_20.csv')

# Drop GEE internal columns
df = df.drop(columns=['system:index', 'count', 'label', 'fresh_proportion',
                       'coca_proportion', 'regional_activity'], errors='ignore')

print(f'Total alert sites: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
df.head()

  Using cached pandas-3.0.2-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached matplotlib-3.10.9-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached contourpy-1.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (117 kB)
  Using cached kiwisolver-1.5.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
  Using cached pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached pandas-3.0.2-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)
Using cached matplotlib-3.10.9-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (8.8 MB)
Using cached contourpy-1.3.

FileNotFoundError: [Errno 2] No such file or directory: 'alert_sites_madre_de_dios_2026_04_20.csv'

## 1. Threat Level Distribution

In [ ]:
threat_order  = ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']
threat_colors = ['#d32f2f', '#f57c00', '#fbc02d', '#388e3c']

counts = df['threat_level'].value_counts().reindex(threat_order)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(threat_order, counts.values, color=threat_colors, edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('Threat Level Distribution\nMadre de Dios Protected Areas — 200 RADD Alert Sites',
             fontsize=13, pad=12)
ax.set_xlabel('Threat Level', fontsize=11)
ax.set_ylabel('Number of Alert Sites', fontsize=11)
ax.set_ylim(0, counts.max() + 10)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig1_threat_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Active/recent illegal activity (CRITICAL + HIGH): {counts["CRITICAL"] + counts["HIGH"]} sites ({(counts["CRITICAL"] + counts["HIGH"])/len(df)*100:.1f}%)')

## 2. Classification Breakdown

In [ ]:
class_labels = {0: 'Intact forest', 1: 'Historical\ndisturbance',
                2: 'Active\ndisturbance', 3: 'Coca\ncultivation'}
class_colors = ['#2e7d32', '#8d6e63', '#c62828', '#7b1fa2']

cls_counts = df['classification'].value_counts().sort_index()
labels = [class_labels[i] for i in cls_counts.index]
colors = [class_colors[i] for i in cls_counts.index]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
bars = ax1.bar(labels, cls_counts.values, color=colors, edgecolor='white')
for bar, val in zip(bars, cls_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')
ax1.set_title('Classification Results by Class', fontsize=12)
ax1.set_ylabel('Number of Alert Sites', fontsize=10)
ax1.spines[['top', 'right']].set_visible(False)

# Pie chart
ax2.pie(cls_counts.values, labels=labels, colors=colors,
        autopct='%1.1f%%', startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax2.set_title('Classification Proportions', fontsize=12)

plt.suptitle('SAR Classification of RADD Alert Sites — Madre de Dios', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fig2_classification_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Active Proportion Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(df['active_proportion'], bins=20, color='#c62828', edgecolor='white',
        alpha=0.85, linewidth=0.8)

ax.axvline(0.5, color='#f57c00', linewidth=2, linestyle='--', label='HIGH threshold (0.5)')
ax.axvline(0.7, color='#d32f2f', linewidth=2, linestyle='--', label='CRITICAL threshold (0.7)')

ax.set_title('Distribution of Active Proportion\n(fraction of pixels classified as active disturbance per alert site)',
             fontsize=12)
ax.set_xlabel('Active Proportion', fontsize=11)
ax.set_ylabel('Number of Alert Sites', fontsize=11)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig3_active_proportion.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Sites with active_proportion > 0.5: {(df['active_proportion'] > 0.5).sum()}")
print(f"Sites with active_proportion > 0.7: {(df['active_proportion'] > 0.7).sum()}")

## 4. Pixel Composition per Threat Level

In [ ]:
pixel_cols = ['n_forest', 'n_historical', 'n_active', 'n_coca']
pixel_labels = ['Intact forest', 'Historical disturbance', 'Active disturbance', 'Coca cultivation']
pixel_colors = ['#2e7d32', '#8d6e63', '#c62828', '#7b1fa2']

grouped = df.groupby('threat_level')[pixel_cols].mean().reindex(threat_order)

fig, ax = plt.subplots(figsize=(10, 6))

bottom = np.zeros(len(threat_order))
for col, label, color in zip(pixel_cols, pixel_labels, pixel_colors):
    vals = grouped[col].values
    ax.bar(threat_order, vals, bottom=bottom, label=label, color=color, edgecolor='white')
    bottom += vals

ax.set_title('Mean Pixel Composition per Threat Level\n(average number of SAR pixels per class within 400m buffer)',
             fontsize=12)
ax.set_xlabel('Threat Level', fontsize=11)
ax.set_ylabel('Mean Pixel Count', fontsize=11)
ax.legend(loc='upper right', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig4_pixel_composition.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Spatial Distribution of Alert Sites

In [ ]:
# Extract coordinates from .geo column
def extract_coords(geo_str):
    try:
        geo = json.loads(geo_str)
        coords = geo['coordinates']
        return coords[0], coords[1]
    except:
        return None, None

df[['lon', 'lat']] = df['.geo'].apply(lambda x: pd.Series(extract_coords(x)))
df_geo = df.dropna(subset=['lon', 'lat'])

threat_color_map = {'CRITICAL': '#d32f2f', 'HIGH': '#f57c00',
                    'MEDIUM': '#fbc02d', 'LOW': '#388e3c'}

fig, ax = plt.subplots(figsize=(10, 8))

for threat in threat_order:
    subset = df_geo[df_geo['threat_level'] == threat]
    ax.scatter(subset['lon'], subset['lat'],
               c=threat_color_map[threat], label=f'{threat} (n={len(subset)})',
               s=30, alpha=0.8, edgecolors='none', zorder=3)

ax.set_title('Spatial Distribution of RADD Alert Sites by Threat Level\nMadre de Dios Focus Region',
             fontsize=12)
ax.set_xlabel('Longitude', fontsize=10)
ax.set_ylabel('Latitude', fontsize=10)
ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
ax.set_facecolor('#f0f0f0')
ax.grid(True, alpha=0.3, color='white')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig5_spatial_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary Statistics Table

In [ ]:
summary = pd.DataFrame({
    'Threat Level': threat_order,
    'Count': [len(df[df['threat_level'] == t]) for t in threat_order],
    'Percentage': [f"{len(df[df['threat_level']==t])/len(df)*100:.1f}%" for t in threat_order],
    'Mean active_proportion': [df[df['threat_level']==t]['active_proportion'].mean() for t in threat_order],
    'Mean n_pixels': [df[df['threat_level']==t]['n_pixels'].mean() for t in threat_order],
})

summary['Mean active_proportion'] = summary['Mean active_proportion'].round(3)
summary['Mean n_pixels'] = summary['Mean n_pixels'].round(1)

print('=== RESULTS SUMMARY ===')
print(summary.to_string(index=False))

print(f'\nTotal sites analysed: {len(df)}')
print(f'CRITICAL + HIGH (active/recent): {len(df[df["threat_level"].isin(["CRITICAL","HIGH"])])} ({len(df[df["threat_level"].isin(["CRITICAL","HIGH"])])/len(df)*100:.1f}%)')
print(f'Coca cultivation detections: {len(df[df["classification"]==3])}')